In [1]:
# Install dependencies

In [3]:
!pip install "abc_atlas_access[notebooks] @ git+https://github.com/AllenInstitute/abc_atlas_access.git" --quiet
!pip install anndata --quiet


In [1]:
# import required modules 
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from pathlib import Path
import anndata
import numpy as np
import os
import scanpy as sc
import pandas as pd
import time
import matplotlib.pyplot as plt

In [9]:
# Set local storage path and initialize ABC cache

# Folder where the dataset will be downloaded
download_dir = Path("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data")
download_dir.mkdir(exist_ok=True)

# Initialize dataset access object pointing to the local cache folder
cache = AbcProjectCache.from_cache_dir(download_dir)


In [9]:
# list available directories 

directories = cache.list_directories
directories


['ASAP-PMDBS-10X',
 'ASAP-PMDBS-taxonomy',
 'Allen-CCF-2020',
 'Consensus-WMB-AIBS-10X',
 'Consensus-WMB-Macosko-10X',
 'Consensus-WMB-integrated-taxonomy',
 'HMBA-10xMultiome-BG',
 'HMBA-10xMultiome-BG-Aligned',
 'HMBA-BG-taxonomy-CCN20250428',
 'HMBA-MERSCOPE-H22.30.001-BG',
 'HMBA-MERSCOPE-QM23.50.001-BG',
 'HMBA-Xenium-CJ23.56.004-BG',
 'MERFISH-C57BL6J-638850',
 'MERFISH-C57BL6J-638850-CCF',
 'MERFISH-C57BL6J-638850-imputed',
 'MERFISH-C57BL6J-638850-sections',
 'SEAAD-taxonomy',
 'WHB-10Xv3',
 'WHB-taxonomy',
 'WMB-10X',
 'WMB-10XMulti',
 'WMB-10Xv2',
 'WMB-10Xv3',
 'WMB-neighborhoods',
 'WMB-taxonomy',
 'Zeng-Aging-Mouse-10Xv3',
 'Zeng-Aging-Mouse-WMB-taxonomy',
 'Zhuang-ABCA-1',
 'Zhuang-ABCA-1-CCF',
 'Zhuang-ABCA-2',
 'Zhuang-ABCA-2-CCF',
 'Zhuang-ABCA-3',
 'Zhuang-ABCA-3-CCF',
 'Zhuang-ABCA-4',
 'Zhuang-ABCA-4-CCF',
 'mmc-gene-mapper']

In [11]:
# List the files available in the 10Xv3 folder
v3_files = cache.list_expression_matrix_files("WMB-10Xv3")
v3_files


['WMB-10Xv3-CB/log2',
 'WMB-10Xv3-CB/raw',
 'WMB-10Xv3-CTXsp/log2',
 'WMB-10Xv3-CTXsp/raw',
 'WMB-10Xv3-HPF/log2',
 'WMB-10Xv3-HPF/raw',
 'WMB-10Xv3-HY/log2',
 'WMB-10Xv3-HY/raw',
 'WMB-10Xv3-Isocortex-1/log2',
 'WMB-10Xv3-Isocortex-1/raw',
 'WMB-10Xv3-Isocortex-2/log2',
 'WMB-10Xv3-Isocortex-2/raw',
 'WMB-10Xv3-MB/log2',
 'WMB-10Xv3-MB/raw',
 'WMB-10Xv3-MY/log2',
 'WMB-10Xv3-MY/raw',
 'WMB-10Xv3-OLF/log2',
 'WMB-10Xv3-OLF/raw',
 'WMB-10Xv3-P/log2',
 'WMB-10Xv3-P/raw',
 'WMB-10Xv3-PAL/log2',
 'WMB-10Xv3-PAL/raw',
 'WMB-10Xv3-STR/log2',
 'WMB-10Xv3-STR/raw',
 'WMB-10Xv3-TH/log2',
 'WMB-10Xv3-TH/raw']

In [13]:
# filter by raw and log2 normalized files 

raw_files = [f for f in v3_files if "raw" in f.lower()]
log_files = [f for f in v3_files if ("log" in f.lower() or "normalized" in f.lower())]

raw_files, log_files


(['WMB-10Xv3-CB/raw',
  'WMB-10Xv3-CTXsp/raw',
  'WMB-10Xv3-HPF/raw',
  'WMB-10Xv3-HY/raw',
  'WMB-10Xv3-Isocortex-1/raw',
  'WMB-10Xv3-Isocortex-2/raw',
  'WMB-10Xv3-MB/raw',
  'WMB-10Xv3-MY/raw',
  'WMB-10Xv3-OLF/raw',
  'WMB-10Xv3-P/raw',
  'WMB-10Xv3-PAL/raw',
  'WMB-10Xv3-STR/raw',
  'WMB-10Xv3-TH/raw'],
 ['WMB-10Xv3-CB/log2',
  'WMB-10Xv3-CTXsp/log2',
  'WMB-10Xv3-HPF/log2',
  'WMB-10Xv3-HY/log2',
  'WMB-10Xv3-Isocortex-1/log2',
  'WMB-10Xv3-Isocortex-2/log2',
  'WMB-10Xv3-MB/log2',
  'WMB-10Xv3-MY/log2',
  'WMB-10Xv3-OLF/log2',
  'WMB-10Xv3-P/log2',
  'WMB-10Xv3-PAL/log2',
  'WMB-10Xv3-STR/log2',
  'WMB-10Xv3-TH/log2'])

In [15]:
# creating separate folders for raw files and log2 files  

raw_dir = download_dir / "raw"

log_dir = download_dir / "log2"

raw_dir.mkdir(exist_ok=True)

log_dir.mkdir(exist_ok=True)


In [17]:
# download raw files 

for f in raw_files:
    print(f"Downloading RAW matrix: {f}")
    cache.get_file_path("WMB-10Xv3", f)


WMB-10Xv3-CB-raw.h5ad: 100%|███████████████████████████████████████████████████████████████████| 5.61G/5.61G [39:42<00:00, 2.35MMB/s]


WMB-10Xv3-CTXsp-raw.h5ad: 100%|████████████████████████████████████████████████████████████████| 3.28G/3.28G [44:26<00:00, 1.23MMB/s]


WMB-10Xv3-HPF-raw.h5ad: 100%|██████████████████████████████████████████████████████████████████| 7.41G/7.41G [39:50<00:00, 3.10MMB/s]


WMB-10Xv3-HY-raw.h5ad: 100%|███████████████████████████████████████████████████████████████████| 7.25G/7.25G [28:36<00:00, 4.22MMB/s]


WMB-10Xv3-Isocortex-1-raw.h5ad: 100%|██████████████████████████████████████████████████████████| 11.8G/11.8G [35:42<00:00, 5.49MMB/s]


...

WMB-10Xv3-OLF-raw.h5ad: 100%|██████████████████████████████████████████████████████████████████| 3.11G/3.11G [04:35<00:00, 11.3MMB/s]


WMB-10Xv3-P-raw.h5ad: 100%|████████████████████████████████████████████████████████████████████| 5.20G/5.20G [06:55<00:00, 12.5MMB/s]


WMB-10Xv3-PAL-raw.h5ad: 100%|██████████████████████████████████████████████████████████████████| 4.07G/4.07G [16:49<00:00, 4.03MMB/s]


WMB-10Xv3-STR-raw.h5ad: 100%|██████████████████████████████████████████████████████████████████| 11.9G/11.9G [26:05<00:00, 7.61MMB/s]


WMB-10Xv3-TH-raw.h5ad: 100%|███████████████████████████████████████████████████████████████████| 5.81G/5.81G [26:47<00:00, 3.62MMB/s]


In [20]:
# download log2 normalized files 

for f in log_files:
    print(f"Downloading LOG2-normalized matrix: {f}")
    cache.get_file_path("WMB-10Xv3", f)


WMB-10Xv3-CB-log2.h5ad: 100%|██████████████████████████████████████████████████████████████████| 5.61G/5.61G [12:26<00:00, 7.52MMB/s]


WMB-10Xv3-CTXsp-log2.h5ad: 100%|███████████████████████████████████████████████████████████████| 3.28G/3.28G [05:50<00:00, 9.34MMB/s]


WMB-10Xv3-HPF-log2.h5ad: 100%|█████████████████████████████████████████████████████████████████| 7.41G/7.41G [26:05<00:00, 4.73MMB/s]


WMB-10Xv3-HY-log2.h5ad: 100%|██████████████████████████████████████████████████████████████████| 7.25G/7.25G [15:29<00:00, 7.80MMB/s]


WMB-10Xv3-Isocortex-1-log2.h5ad: 100%|█████████████████████████████████████████████████████████| 11.8G/11.8G [17:11<00:00, 11.4MMB/s]


...

WMB-10Xv3-OLF-log2.h5ad: 100%|█████████████████████████████████████████████████████████████████| 3.11G/3.11G [15:34<00:00, 3.33MMB/s]


WMB-10Xv3-P-log2.h5ad: 100%|███████████████████████████████████████████████████████████████████| 5.20G/5.20G [20:50<00:00, 4.16MMB/s]


WMB-10Xv3-PAL-log2.h5ad: 100%|█████████████████████████████████████████████████████████████████| 4.07G/4.07G [16:14<00:00, 4.18MMB/s]


WMB-10Xv3-STR-log2.h5ad: 100%|█████████████████████████████████████████████████████████████████| 11.9G/11.9G [38:16<00:00, 5.19MMB/s]


WMB-10Xv3-TH-log2.h5ad: 100%|██████████████████████████████████████████████████████████████████| 5.81G/5.81G [10:56<00:00, 8.85MMB/s]


In [21]:
# SANITY CHECK (optional): Load the first log2-normalized file to confirm everything worked
test_file = cache.get_file_path("WMB-10Xv3", log_files[0])

adata = anndata.read_h5ad(test_file)

adata

AnnData object with n_obs × n_vars = 182026 × 32285
    obs: 'cell_barcode', 'library_label', 'anatomical_division_label'
    var: 'gene_symbol'
    uns: 'normalization', 'parent', 'parent_layer', 'parent_rows'

In [10]:
# looking at available metadata files  
abc_cache.list_metadata_files('WMB-10X')

['cell_metadata',
 'cell_metadata_with_cluster_annotation',
 'example_genes_all_cells_expression',
 'gene',
 'region_of_interest_metadata']

In [12]:
# downloading metadata

cell = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print("Number of cells = ", len(cell))
cell.head(5)


Number of cells =  4042976


,cell_barcode,barcoded_cell_sample_label,library_label,feature_matrix_label,entity,brain_section_label,library_method,region_of_interest_acronym,donor_label,donor_genotype,donor_sex,dataset_label,x,y,cluster_alias,abc_sample_id
cell_label,,,,,,,,,,,,,,,,
GCGAGAAGTTAAGGGC-410_B05,GCGAGAAGTTAAGGGC,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.146826,-3.086639,1,484be5df-5d44-4bfe-9652-7b5bc739c211
AATGGCTCAGCTCCTT-411_B06,AATGGCTCAGCTCCTT,411_B06,L8TX_201029_01_E10,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550851,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.138481,-3.022000,1,5638505d-e1e8-457f-9e5b-59e3e2302417
AACACACGTTGCTTGA-410_B05,AACACACGTTGCTTGA,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.472557,-2.992709,1,a0544e29-194f-4d34-9af4-13e7377b648f
CACAGATAGAGGCGGA-410_A05,CACAGATAGAGGCGGA,410_A05,L8TX_201029_01_A10,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.379622,-3.043442,1,c777ac0b-77e1-4d76-bf8e-2b3d9e08b253
AAAGTGAAGCATTTCG-410_B05,AAAGTGAAGCATTTCG,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.909480,-2.601536,1,49860925-e82b-46df-a228-fd2f97e75d39


In [13]:
# downloading metadata with cluster info

cell = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='cell_metadata_with_cluster_annotation',
    dtype={'cell_label': str}
)

cell.set_index('cell_label', inplace=True)

print("Number of cells = ", len(cell))

cell.head(5)

cell_metadata_with_cluster_annotation.csv: 100%|█| 1.39G/1.39G [08:38<00:00, 2.6
/Users/cclu223/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/abc_atlas_access/abc_atlas_cache/abc_project_cache.py:643: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, **kwargs)


Number of cells =  4042976


,cell_barcode,barcoded_cell_sample_label,library_label,feature_matrix_label,entity,brain_section_label,library_method,region_of_interest_acronym,donor_label,donor_genotype,...,subclass,supertype,cluster,neurotransmitter_color,class_color,subclass_color,supertype_color,cluster_color,region_of_interest_order,region_of_interest_color
cell_label,,,,,,,,,,,,,,,,,,,,,
GCGAGAAGTTAAGGGC-410_B05,GCGAGAAGTTAAGGGC,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,...,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,0326 L2 IT PPP-APr Glut_3,#2B93DF,#FA0087,#0F6632,#266DFF,#64661F,15,#CCB05C
AATGGCTCAGCTCCTT-411_B06,AATGGCTCAGCTCCTT,411_B06,L8TX_201029_01_E10,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550851,Ai14(RCL-tdT)/wt,...,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,0326 L2 IT PPP-APr Glut_3,#2B93DF,#FA0087,#0F6632,#266DFF,#64661F,15,#CCB05C
AACACACGTTGCTTGA-410_B05,AACACACGTTGCTTGA,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,...,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,0326 L2 IT PPP-APr Glut_3,#2B93DF,#FA0087,#0F6632,#266DFF,#64661F,15,#CCB05C
CACAGATAGAGGCGGA-410_A05,CACAGATAGAGGCGGA,410_A05,L8TX_201029_01_A10,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,...,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,0326 L2 IT PPP-APr Glut_3,#2B93DF,#FA0087,#0F6632,#266DFF,#64661F,15,#CCB05C
AAAGTGAAGCATTTCG-410_B05,AAAGTGAAGCATTTCG,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,...,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,0326 L2 IT PPP-APr Glut_3,#2B93DF,#FA0087,#0F6632,#266DFF,#64661F,15,#CCB05C


In [23]:
# !!DOWNSAMPLING TRIAL RUN!!
# Downsample: keep every 10th cell
subset = adata[np.arange(adata.n_obs) % 10 == 0].copy()
subset

AnnData object with n_obs × n_vars = 18203 × 32285
    obs: 'cell_barcode', 'library_label', 'anatomical_division_label'
    var: 'gene_symbol'
    uns: 'normalization', 'parent', 'parent_layer', 'parent_rows'

In [24]:
# !!DOWNSAMPLING TRIAL RUN!!
# making sure that the downsampling worked 

print("Original cell count:", adata.n_obs)

print("Downsampled cell count:", subset.n_obs)

print("Fraction retained:", subset.n_obs / adata.n_obs)

Original cell count: 182026
Downsampled cell count: 18203
Fraction retained: 0.10000219748827091


/var/folders/j3/mrjbghwj6g9_vh4vwr03ppt00000gr/T/ipykernel_8496/3068254324.py:25: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  cell_meta = pd.read_csv(METADATA_FILE, index_col=0)



Processing: WMB-10Xv3-OLF-raw.h5ad
  Neuronal cap: 01 IT-ET Glut → 1000
  Neuronal cap: 05 OB-IMN GABA → 1000
  Neuronal downsample: 06 CTX-CGE GABA → 120

Processing: WMB-10Xv3-TH-raw.h5ad
  Neuronal downsample: 12 HY GABA → 663
  Neuronal downsample: 17 MH-LH Glut → 603
  Neuronal cap: 18 TH Glut → 1000
  Neuronal downsample: 19 MB Glut → 648
  Neuronal downsample: 20 MB GABA → 430

Processing: WMB-10Xv3-CTXsp-raw.h5ad
  Neuronal cap: 01 IT-ET Glut → 1000
  Neuronal downsample: 02 NP-CT-L6b Glut → 308
  Neuronal downsample: 06 CTX-CGE GABA → 228
  Neuronal downsample: 07 CTX-MGE GABA → 273
  Neuronal downsample: 09 CNU-LGE GABA → 446
  Neuronal downsample: 13 CNU-HYa Glut → 121

Processing: WMB-10Xv3-Isocortex-1-raw.h5ad
  Neuronal cap: 01 IT-ET Glut → 1000
  Neuronal cap: 02 NP-CT-L6b Glut → 1000
  Neuronal cap: 06 CTX-CGE GABA → 1000
  Neuronal cap: 07 CTX-MGE GABA → 1000

Processing: WMB-10Xv3-HY-raw.h5ad
  Neuronal downsample: 08 CNU-MGE GABA → 185
  Neuronal downsample: 11 CNU-

/var/folders/j3/mrjbghwj6g9_vh4vwr03ppt00000gr/T/ipykernel_8496/3774982726.py:9: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  cell = pd.read_csv("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/metadata/WMB-10X/20241115/views/cell_metadata_with_cluster_annotation.csv")


Number of cells =  4042976


,class,cell_count_original,cell_count_downsampled
0,01 IT-ET Glut,1095484,5888
1,31 OPC-Oligo,545179,73073
2,02 NP-CT-L6b Glut,310198,3534
3,30 Astro-Epen,308681,50148
4,29 CB Glut,141106,2744
5,06 CTX-CGE GABA,139032,3760
6,33 Vascular,137493,40162
7,07 CTX-MGE GABA,122085,4032
8,19 MB Glut,120552,3010
9,18 TH Glut,115401,2211


,subclass,cell_count_original,cell_count_downsampled
0,327 Oligo NN,422574,58259
1,006 L4/5 IT CTX Glut,369221,874
2,030 L6 CT CTX Glut,201912,1886
3,007 L2/3 IT CTX Glut,172213,456
4,319 Astro-TE NN,154311,22983
...,...,...,...
278,244 MV-SPIV Slc6a2 Glut,467,15
279,112 GPi Tbr1 Cngb3 Gaba-Glut,460,42
280,307 RO-RPA Pkd2l1 Gaba,448,15
281,338 Lymphoid NN,404,361


,subclass,cell_count_original,cell_count_downsampled
0,327 Oligo NN,422574,58259
1,319 Astro-TE NN,154311,22983
2,318 Astro-NT NN,140669,19213
3,326 OPC NN,122605,14814
4,333 Endo NN,88011,16174
5,334 Microglia NN,86232,8554
6,331 Peri NN,24907,5611
7,332 SMC NN,14614,9009
8,330 VLMC NN,9104,8544
9,335 BAM NN,5626,5516


,cluster,cell_count
5284,5285 MOL NN_4,264669
5224,5225 Astro-TE NN_3,131046
5283,5284 MOL NN_4,120642
5268,5269 OPC NN_1,117304
5200,5201 CB Granule Glut_2,115909
...,...,...
4850,4851 MY Lhx1 Gly-Gaba_1,9
4878,4879 MY Lhx1 Gly-Gaba_2,9
4382,4383 PGRN-PARN-MDRN Hoxb5 Glut_5,9
3510,3511 PAG-MRN-RN Foxa2 Gaba_1,9


Subcluster
TE_Astrocytes         22983
MOL_02                22860
NT_Astrocytes         19213
OPC_01                14259
MOL_01                11843
                      ...  
B cells_01               41
Monocytes_01             29
ILC_01                   27
Dendritic Cells_01       26
NFOL_01                   1
Name: count, Length: 68, dtype: int64